# DPFE Email Privacy Attack Experiment

Replication of **Huang et al. (2022)** — *Are Large Pre-Trained Language Models Leaking Your Personal Information?*

Fine-tunes **GPT-2 Large** on ENRON emails using LoRA + DP-SGD, then runs a privacy attack to measure email address leakage at different noise levels.

**Runtime:** ~3–4 hours on an A100, ~8–10 hours on a V100/T4.

> **Before running:** Go to `Runtime → Change runtime type` and select **A100** (or at minimum V100) under Hardware accelerator.

> **Resuming after disconnect:** Enable Google Drive in Cell 3 (`USE_DRIVE = True`). The experiment saves a checkpoint after each noise level completes — re-running the notebook will skip already-finished runs automatically.

In [1]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type and select a GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Tue Jun  2 06:41:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ── 2. Clone repo and install dependencies ───────────────────────────────────
import os

REPO_URL = 'https://github.com/usffish/dpfe-email-privacy-experiment.git'
BRANCH   = 'colab'
WORK_DIR = '/content/dpfe-email-privacy-experiment'

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!pip install -q -r requirements.txt

Cloning into '/content/dpfe-email-privacy-experiment'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 134 (delta 89), reused 97 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 60.09 KiB | 10.02 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/dpfe-email-privacy-experiment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 139.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

In [3]:
# ── 3. Mount Google Drive for persistence and disconnect recovery ──────────────
# Strongly recommended: saves results and the model cache to Drive so that if
# Colab disconnects, re-running the notebook resumes from the last completed run.
# Set USE_DRIVE = False only if you are certain the session won't disconnect.

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/dpfe-experiment'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    # Symlink results and model cache into Drive so they survive session resets
    os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/hf_cache', exist_ok=True)
    if not os.path.islink(f'{WORK_DIR}/results'):
        os.symlink(f'{DRIVE_DIR}/results', f'{WORK_DIR}/results')
    os.environ['HF_HOME'] = f'{DRIVE_DIR}/hf_cache'
    print(f'Drive mounted. Results → {DRIVE_DIR}/results')
    print('Checkpoint will be saved after each noise level run.')
else:
    os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
    print('Running without Drive. Results will be lost if the session disconnects.')

Mounted at /content/drive
Drive mounted. Results → /content/drive/MyDrive/dpfe-experiment/results
Checkpoint will be saved after each noise level run.


In [4]:
# ── 4. Download ENRON corpus ──────────────────────────────────────────────────
# The real corpus is required — the script will raise an error if it is missing.
# Download takes ~5 minutes.

import os
enron_dir = f'{WORK_DIR}/enron_data'
if not os.path.exists(f'{enron_dir}/maildir'):
    print('Downloading ENRON corpus (~432 MB)...')
    !wget -q https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz -O /tmp/enron.tar.gz
    !mkdir -p {enron_dir}
    !tar -xzf /tmp/enron.tar.gz -C {enron_dir}
    print('Done.')
else:
    print('ENRON corpus already present.')

^C

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
Done.


In [ ]:
# ── 5. Configuration ─────────────────────────────────────────────────────────
# Override any CONFIG values here before running the experiment.
# These are written to a .env file that main.py picks up via python-dotenv.
#
# Smoke-test config (fast validation, ~15 min on A100):
#   MAX_EMAILS=3000, EPOCHS=1, SUBSET_PAIRS=200
# Full experiment config (replicates Table 11, ~3–4 h on A100):
#   MAX_EMAILS=50000, EPOCHS=3, SUBSET_PAIRS=3238
#
# BATCH_SIZE vs DP_BATCH_SIZE:
#   BATCH_SIZE applies to the σ=0 baseline (standard SGD).
#   DP_BATCH_SIZE applies to all Opacus DP-SGD runs (σ > 0).
#   Opacus stores per-sample gradients for every sample simultaneously,
#   roughly doubling backprop memory. At float32 + max_length=256,
#   batch=16 OOMs a 40 GB A100; 8 fits comfortably.

env_config = {
    'MODEL_NAME':    'gpt2-large',
    'BATCH_SIZE':    '16',
    'DP_BATCH_SIZE': '8',
    'EPOCHS':        '3',
    'MAX_EMAILS':    '50000',
    'SUBSET_PAIRS':  '3238',
    'LEARNING_RATE': '5e-5',
    'SEED':          '42',
}

with open(f'{WORK_DIR}/.env', 'w') as f:
    for k, v in env_config.items():
        f.write(f'{k}={v}\n')

print('Config written to .env:')
for k, v in env_config.items():
    print(f'  {k}={v}')

In [ ]:
# ── 5b. Clean re-run (OPTIONAL — skip unless you want to discard prior results) ──
# Run this cell only if you want to start the experiment from scratch.
# Required when: changing config mid-experiment, applying a main.py fix to
# already-completed runs, or switching from smoke-test to full-run values.
#
# WARNING: this permanently deletes the results checkpoint from Google Drive.
# Completed noise levels will NOT be skipped — all 5 runs will re-execute.

CLEAN_RERUN = False  # ← set to True to wipe the checkpoint

if CLEAN_RERUN:
    import os
    results_file = f'{WORK_DIR}/results/table_11_results.json'
    if os.path.exists(results_file):
        os.remove(results_file)
        print(f'Deleted {results_file} — experiment will start from scratch.')
    else:
        print('No checkpoint found — nothing to delete.')
else:
    print('Clean re-run skipped (CLEAN_RERUN = False).')

In [ ]:
# ── 6. Run experiment ─────────────────────────────────────────────────────────
# Streams output live. Expect ~3-4h on A100, ~8-10h on V100/T4.
# If Colab disconnects: re-run all cells from the top — completed noise levels
# are loaded from the Drive checkpoint and skipped automatically.

!python -u main.py

2026-06-02 06:42:42.793113: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 06:42:42.865626: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DPFE Email Privacy Attack Experiment
Model: gpt2-large — LoRA (float16)
LoRA: r=16, alpha=32, targets=['c_attn']
Device: cuda

[Step 1] Loading and processing ENRON email data...
Processing ENRON email corpus...
Scanning ENRON corpus: 9905file [00:02, 4744.86file/s, bodies=9570, pairs=335]
  Training emails: 9570
  Attack pairs: 146
tokenizer_config.json: 100% 26.0/26.0 [

In [ ]:
# ── 7. Display results ────────────────────────────────────────────────────────
import json
import os

results_path = f'{WORK_DIR}/results/table_11_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print('\nResults saved to:', results_path)
    if USE_DRIVE:
        print('Also persisted to Google Drive.')
else:
    print('Results file not found — experiment may still be running or encountered an error.')